# T1 and T2 decay fits

Load the T1 and Ramsey bundles from `data/t12`, fit exponential decay models, and report the fitted lifetimes and decay rates.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

from presentation_style import polish_axes, use_presentation_style

use_presentation_style()

data_dir = Path("data/t12")
t1_path = next(data_dir.glob("05_T1_*_data_bundle.npz"))
t2_path = next(data_dir.glob("06a_ramsey_*_data_bundle.npz"))

print(f"T1 bundle: {t1_path}")
print(f"T2/Ramsey bundle: {t2_path}")

In [ ]:
def t1_decay(t_us, offset, amplitude, tau_us):
    return offset + amplitude * np.exp(-t_us / tau_us)


def ramsey_decay(t_us, offset, amplitude, tau_us, frequency_mhz, phase_rad):
    return offset + amplitude * np.exp(-t_us / tau_us) * np.cos(
        2 * np.pi * frequency_mhz * t_us + phase_rad
    )


def fit_t1(t_us, y):
    tail = np.mean(y[-10:])
    p0 = [tail, y[0] - tail, 0.5 * (t_us.max() - t_us.min())]
    bounds = ([0.0, -1.2, 0.01], [1.2, 1.2, 10_000.0])
    popt, pcov = curve_fit(t1_decay, t_us, y, p0=p0, bounds=bounds, maxfev=20_000)
    return popt, np.sqrt(np.diag(pcov))


def fit_ramsey(t_us, y, frequency_guess_mhz=2.0):
    p0 = [np.mean(y), 0.5 * (y.max() - y.min()), 0.5 * t_us.max(), frequency_guess_mhz, 0.0]
    bounds = ([0.0, -1.0, 0.01, 0.0, -2 * np.pi], [1.2, 1.0, 10_000.0, 10.0, 2 * np.pi])
    popt, pcov = curve_fit(ramsey_decay, t_us, y, p0=p0, bounds=bounds, maxfev=100_000)
    return popt, np.sqrt(np.diag(pcov))


def decay_rate_khz(tau_us):
    # gamma = 1 / tau. Because tau is in microseconds, gamma[1/us] = gamma[MHz].
    return 1_000 / tau_us

In [ ]:
t1_bundle = np.load(t1_path, allow_pickle=True)
t2_bundle = np.load(t2_path, allow_pickle=True)

t1_time_us = t1_bundle["data__idle_time"] / 1_000
t1_state = t1_bundle["data__state"][0]
t1_fit, t1_err = fit_t1(t1_time_us, t1_state)

t2_time_us = t2_bundle["data__idle_time"] / 1_000
t2_state_all = t2_bundle["data__state"][0]
detuning_signs = t2_bundle["data__detuning_signs"]
t2_sign = 1
t2_idx = int(np.where(detuning_signs == t2_sign)[0][0])
t2_state = t2_state_all[:, t2_idx]
t2_fit, t2_err = fit_ramsey(t2_time_us, t2_state)

print(f"T1 = {t1_fit[2]:.2f} +/- {t1_err[2]:.2f} us")
print(f"T1 decay rate = {decay_rate_khz(t1_fit[2]):.2f} kHz")

print(
    f"T2* sign {t2_sign:+d}: {t2_fit[2]:.2f} +/- {t2_err[2]:.2f} us, "
    f"decay rate = {decay_rate_khz(t2_fit[2]):.2f} kHz, "
    f"Ramsey frequency = {t2_fit[3]:.3f} MHz"
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8.8, 3.5), constrained_layout=True)

t1_dense = np.linspace(t1_time_us.min(), t1_time_us.max(), 600)
axes[0].plot(t1_time_us, t1_state, "o", ms=5.2, color="#2f6f9f")
axes[0].plot(
    t1_dense,
    t1_decay(t1_dense, *t1_fit),
    color="#d1495b",
    lw=3.3,
    label=f"T1 = {t1_fit[2]:.1f} us",
)
axes[0].set_xlabel("Idle time (us)")
axes[0].set_ylabel("Excited population")
axes[0].set_title("")
axes[0].legend(fontsize=16, handlelength=1.3, loc="upper right")
polish_axes(axes[0])

t2_dense = np.linspace(t2_time_us.min(), t2_time_us.max(), 1_200)
axes[1].plot(t2_time_us, t2_state, "o", ms=4.8, color="#2a9d8f", alpha=0.72)
axes[1].plot(
    t2_dense,
    ramsey_decay(t2_dense, *t2_fit),
    color="#7b2cbf",
    lw=3.0,
    label=f"T2* = {t2_fit[2]:.1f} us",
)

axes[1].set_xlabel("Idle time (us)")
axes[1].set_ylabel("")
axes[1].set_title("")
axes[1].legend(fontsize=16, handlelength=1.3, loc="upper right")
polish_axes(axes[1])

for ax in axes:
    ax.set_ylim(0.08, 1.05)
    ax.title.set_size(18)
    ax.xaxis.label.set_size(18)
    ax.yaxis.label.set_size(18)
    ax.tick_params(labelsize=16)

fig.savefig("figures/07_t1_t2_decay_fits.png", dpi=300)
plt.show()